# Nhận diện biển báo giao thông Việt Nam (YOLOv8 / Faster R-CNN)

Notebook chạy được trên **Google Colab** và **Kaggle**. Bật GPU trước khi chạy:
- Colab: *Runtime → Change runtime type → T4 GPU*
- Kaggle: *Settings → Accelerator → GPU*

Chạy lần lượt từ trên xuống. Mục 4 (Faster R-CNN) là tùy chọn.

## 1. Cài đặt

In [ ]:
!pip install -q ultralytics
!yolo settings sync=False

import os, sys, glob, torch
IN_COLAB = 'google.colab' in sys.modules
print('Colab' if IN_COLAB else 'Kaggle/khác', '| GPU:', torch.cuda.is_available())

## 2. Lấy trọng số mô hình

Mặc định clone repo GitHub của đồ án. Nếu không muốn clone, hãy upload thư mục `YOLOV8/` và
`Faster R-CNN MobileNetV3/` vào `/content` (Colab) hoặc dataset (Kaggle) rồi sửa `REPO_DIR`.
Trên Kaggle cần bật *Settings → Internet* để clone.

In [ ]:
REPO_URL = 'https://github.com/thuanvipghe/Traffic-sign-VietNam-recognition'
REPO_DIR = 'Traffic-sign-VietNam-recognition'

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

YOLO_WEIGHTS = {
    'n': f'{REPO_DIR}/YOLOV8/yolov8n/yolov8n_final.pt',
    's': f'{REPO_DIR}/YOLOV8/yolov8s/yolov8s_final.pt',
    'm': f'{REPO_DIR}/YOLOV8/yolov8m/yolov8m_final.pt',
}
FRCNN_WEIGHTS = f'{REPO_DIR}/Faster R-CNN MobileNetV3/faster_epoch_final.pth'
for k, v in {**YOLO_WEIGHTS, 'frcnn': FRCNN_WEIGHTS}.items():
    print(k, os.path.exists(v), v)

## 3. YOLOv8

Chọn bản `n` (nhanh nhất), `s` (cân bằng) hoặc `m` (chính xác nhất).

In [ ]:
from ultralytics import YOLO

MODEL_SIZE = 'm'   # 'n' | 's' | 'm'
CONF = 0.25

yolo = YOLO(YOLO_WEIGHTS[MODEL_SIZE])
print(len(yolo.names), 'lớp:', list(yolo.names.values())[:10], '...')

def get_input(exts):
    '''Colab: mở hộp thoại upload. Kaggle: dùng đường dẫn đặt ở INPUT_PATH.'''
    if IN_COLAB:
        from google.colab import files
        up = files.upload()
        return list(up.keys())[0]
    return INPUT_PATH

INPUT_PATH = '/kaggle/input/ten-dataset/anh.jpg'   # chỉ dùng khi chạy trên Kaggle

### 3a. Ảnh

In [ ]:
from IPython.display import display
from PIL import Image

img_path = get_input(('.jpg', '.jpeg', '.png'))
res = yolo.predict(img_path, conf=CONF, verbose=False)[0]
for b in res.boxes:
    print(yolo.names[int(b.cls)], f'{float(b.conf):.2f}')
display(Image.fromarray(res.plot()[..., ::-1]))   # plot() trả về BGR

### 3b. Video

Video dài sẽ chạy lâu. Nên thử với đoạn ngắn (vài chục giây) trước.
Dùng `track` (ByteTrack) như `main.py` gốc; đổi thành `predict` nếu không cần theo dõi.

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import subprocess

vid_path = get_input(('.mp4', '.mov', '.avi'))
r = yolo.track(vid_path, conf=CONF, tracker='bytetrack.yaml', save=True, show=False, stream=False, verbose=False)
save_dir = r[0].save_dir
out_avi = glob.glob(os.path.join(save_dir, '*'))[0]
out_mp4 = '/tmp/result.mp4'
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', out_avi, '-vcodec', 'libx264', out_mp4], check=True)
data = b64encode(open(out_mp4, 'rb').read()).decode()
HTML(f'<video width=720 controls src="data:video/mp4;base64,{data}"></video>')

### 3c. Đo FPS

In [ ]:
import time, numpy as np
dummy = (np.random.rand(1080, 1920, 3) * 255).astype('uint8')
for _ in range(5): yolo.predict(dummy, verbose=False)          # khởi động
t = time.time()
for _ in range(50): yolo.predict(dummy, verbose=False)
print(f'~{50 / (time.time() - t):.1f} FPS (gồm tiền/hậu xử lý, ảnh 1920x1080)')

## 4. Faster R-CNN MobileNetV3 (tùy chọn)

Đã sửa so với script gốc: `weights_only=True` khi nạp file (tránh chạy mã pickle),
`weights=None` để không tải thêm trọng số COCO (đã có trong checkpoint), và hiển thị mã biển báo thay vì số.

In [ ]:
import torch, torchvision
from PIL import Image, ImageDraw
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms as T

id_to_label = dict(enumerate([
    '102','103a','103b','103c','104','106a','106b','112','117','123a','123b','124a','124c',
    '127.40','127.50','127.60','127.70','127.80','128','130','131a','131b','131c','136','137',
    '201','203','205','207','208','221','224','225','227','239a','239b','245','301','302','303',
    '401','402','423','425','434','443','other1','other12','other13']))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
frcnn = fasterrcnn_mobilenet_v3_large_fpn(weights=None, weights_backbone=None)
frcnn.roi_heads.box_predictor = FastRCNNPredictor(frcnn.roi_heads.box_predictor.cls_score.in_features, 50)
frcnn.load_state_dict(torch.load(FRCNN_WEIGHTS, map_location=device, weights_only=True))
frcnn.to(device).eval()

@torch.no_grad()
def frcnn_predict(image, threshold=0.5):
    x = T.ToTensor()(image).to(device)
    p = frcnn([x])[0]
    draw = ImageDraw.Draw(image)
    for box, lab, sc in zip(p['boxes'].cpu(), p['labels'].cpu(), p['scores'].cpu()):
        if sc < threshold: continue
        x1, y1, x2, y2 = box.tolist()
        draw.rectangle([x1, y1, x2, y2], outline='red', width=3)
        draw.text((x1, max(0, y1 - 12)), f"{id_to_label.get(int(lab), '?')} {sc:.2f}", fill='red')
    return image

img_path = get_input(('.jpg', '.jpeg', '.png'))
display(frcnn_predict(Image.open(img_path).convert('RGB')))

## Lưu ý

- Mô hình được huấn luyện trên ảnh camera hành trình 1920x1080 ở TP.HCM, nên ảnh khác góc nhìn có thể kém chính xác.
- Chỉ nhận diện được 49 loại biển trong danh sách; biển khác sẽ bị bỏ qua hoặc nhận nhầm.
- Webcam không dùng trực tiếp được trên Colab/Kaggle. Muốn real-time bằng webcam, hãy chạy `yolo predict source=0` trên máy cá nhân.